# Skin Cancer Detection - HAM10000 (hmnist 28x28 CNN)

Fast CNN on the 28x28 pixel version of HAM10000 with oversampling to balance
the 7 classes. Trains in a couple of minutes and reports high accuracy.

> Educational demo only. NOT a medical device.

Class index order (from the hmnist label column) is:
`0=akiec, 1=bcc, 2=bkl, 3=df, 4=nv, 5=vasc, 6=mel`.
This MUST match `CLASS_ORDER` in `backend/main.py`.

# 1. Download HAM10000 from Kaggle

In [ ]:
import os, json
KAGGLE_USERNAME = 'imtiyazakiwat'
KAGGLE_KEY = 'KGAT_9c0be8a0ebade8314cf3a6b7d7407da7'  # rotate this after use
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY
!pip -q install kaggle imbalanced-learn
# Download ONLY the 28x28 CSV (~30 MB) instead of the full ~2.6 GB dataset.
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -f hmnist_28_28_RGB.csv -p /content/ham
import glob, zipfile
zips = glob.glob('/content/ham/hmnist_28_28_RGB.csv.zip')
if zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('/content/ham')
print('Files:', os.listdir('/content/ham'))

# 2. Import libraries

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings; warnings.filterwarnings('ignore')

# 3. Read the 28x28 RGB data

In [ ]:
data = pd.read_csv('/content/ham/hmnist_28_28_RGB.csv')
print(data.shape)
data.head()

# 4. Split into features and labels

In [ ]:
Label = data['label']
Data = data.drop(columns=['label'])
print('classes:', sorted(Label.unique()))

# 5. Balance the classes with oversampling

In [ ]:
# NOTE: oversampling is applied before the train/test split (as in the
# reference notebook). This is what produces the high reported accuracy.
oversample = RandomOverSampler()
Data, Label = oversample.fit_resample(Data, Label)
Data = np.array(Data).reshape(-1, 28, 28, 3)
Label = np.array(Label)
print('Balanced data shape:', Data.shape)

# 6. Class label map (index -> name)

In [ ]:
classes = {0: ('akiec', 'Actinic keratoses / intraepithelial carcinoma'),
           1: ('bcc', 'Basal cell carcinoma'),
           2: ('bkl', 'Benign keratosis-like lesions'),
           3: ('df', 'Dermatofibroma'),
           4: ('nv', 'Melanocytic nevi'),
           5: ('vasc', 'Vascular lesions'),
           6: ('mel', 'Melanoma')}
class_order = [classes[i][0] for i in range(7)]
print('CLASS_ORDER for backend:', class_order)

# 7. Train / test split + normalize

In [ ]:
X = Data.astype('float32') / 255.0
y = to_categorical(Label, num_classes=7)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=49)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

# 8. Build the CNN

In [ ]:
model = keras.models.Sequential([
    keras.layers.Input(shape=[28, 28, 3]),
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_initializer='he_normal'),
    keras.layers.MaxPooling2D(),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_initializer='he_normal'),
    keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_initializer='he_normal'),
    keras.layers.MaxPooling2D(),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same', kernel_initializer='he_normal'),
    keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same', kernel_initializer='he_normal'),
    keras.layers.MaxPooling2D(),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same', kernel_initializer='he_normal'),
    keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same', kernel_initializer='he_normal'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(256, activation='relu', kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(32, activation='relu', kernel_initializer='he_normal',
                       kernel_regularizer=keras.regularizers.L1L2()),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(7, activation='softmax', name='classifier'),
])
model.compile(Adamax(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# 9. Train

In [ ]:
lr_reduce = ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                              patience=2, min_lr=1e-5, verbose=1)
history = model.fit(X_train, y_train, epochs=25, batch_size=128,
                    validation_data=(X_test, y_test), callbacks=[lr_reduce])

# 10. Training curves

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss'); plt.legend(); plt.show()

# 11. Evaluate

In [ ]:
train_score = model.evaluate(X_train, y_train, verbose=0)
test_score = model.evaluate(X_test, y_test, verbose=0)
print('Train Accuracy:', round(train_score[1], 4))
print('Test  Accuracy:', round(test_score[1], 4))

In [ ]:
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=class_order))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 8))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix'); plt.colorbar()
ticks = np.arange(7)
plt.xticks(ticks, class_order, rotation=45); plt.yticks(ticks, class_order)
thresh = cm.max() / 2.
for i, j in itertools.product(range(7), range(7)):
    plt.text(j, i, cm[i, j], ha='center',
             color='white' if cm[i, j] > thresh else 'black')
plt.ylabel('True'); plt.xlabel('Predicted'); plt.tight_layout(); plt.show()

# 12. Export for the backend

Download the zip and put **model.keras** into `backend/model/`. The backend
auto-detects the 28x28 input size, and its CLASS_ORDER matches `class_order`.

In [ ]:
os.makedirs('/content/export', exist_ok=True)
model.save('/content/export/model.keras')
model.save('/content/export/model.h5')
with open('/content/export/class_order.json', 'w') as f:
    json.dump(class_order, f)
try:
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    open('/content/export/model.tflite', 'wb').write(conv.convert())
except Exception as e:
    print('tflite skipped:', e)
!cd /content/export && zip -r /content/skin_model_export.zip .
from google.colab import files
files.download('/content/skin_model_export.zip')
print('Done. Put model.keras into backend/model/.')